# ViFeedback — Kaggle training

**This notebook contains no logic.** It unpacks the repository and calls the same `vifeedback`
CLI the laptop runs, so results land in the identical `results/registry.csv` schema and a
platform-only bug is impossible (docs/ROADMAP.md § 4).

## Before you run anything

| Step | Where |
|---|---|
| 1. Accelerator → **GPU P100** | *Settings* panel, right side |
| 2. Internet → **On** | *Settings* panel — required to reach the HF Hub |
| 3. Attach the repo dataset | *Add Data* → your `vifeedback-repo` dataset |
| 4. *(optional)* `HF_TOKEN` secret | *Add-ons → Secrets* — avoids Hub rate limits |

Full step-by-step, including how to create the dataset: **`docs/KAGGLE_GUIDE.md`**.

## What belongs here

The laptop reference GPU is an RTX 3050 with **4.29 GB**. Measured and calculated fits:

| Model | Params | VRAM | Runs on the laptop? |
|---|---|---|---|
| `phobert-base` | 135M | **3.6 GB measured** | yes — do not run it here |
| `phobert-base-v2` | 135M | ~3.6 GB | yes |
| `visobert` | 97M | ~3.0 GB | yes |
| `xlm-roberta-base` + `--freeze-embeddings` | 85M trainable | ~3.6 GB | yes |
| `xlm-roberta-base` full | 277M | ~5.9 GB | **no → here** |
| `phobert-large` | 368M | ~7.9 GB | **no → here** |
| `CafeBERT` | 560M | ~11 GB | **no → here** |

## What must NOT run here

**Any latency benchmark.** Phase 6 measures CPU p95 on the documented reference machine
(AMD Ryzen 5 6600H, no AVX512-VNNI). A number from a Kaggle VM is not comparable to it and must
not enter the registry.


---

## 1. Verify the environment


In [ ]:
import torch

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

assert torch.cuda.is_available(), (
    'No GPU. Settings -> Accelerator -> GPU P100, then re-run.'
)
p = torch.cuda.get_device_properties(0)
vram = p.total_memory / 1e9
print(f'{p.name}  {vram:.2f} GB  capability {p.major}.{p.minor}  torch {torch.__version__}')
assert vram > 10, f'Only {vram:.1f} GB — not enough for the models this notebook is for.'

import socket

try:
    socket.create_connection(('huggingface.co', 443), timeout=5).close()
    print('internet: OK')
except OSError:
    raise SystemExit('No internet. Settings -> Internet -> On, then re-run.') from None


### Optional: Hugging Face token

Unauthenticated Hub downloads are rate-limited. If you added an `HF_TOKEN` secret
(*Add-ons → Secrets*), this picks it up; if not, it carries on unauthenticated.


In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle Secrets')
except Exception as e:
    print(f'No HF_TOKEN ({type(e).__name__}) — continuing unauthenticated, which is fine')

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['PYTHONUNBUFFERED'] = '1'


---

## 2. Unpack the repository

**Route B (recommended)** reads a Kaggle Dataset you created from `git archive` — versioned, and
the notebook keeps working without internet. **Route A** clones a git remote if you have one.


In [ ]:
import glob
import pathlib
import shutil

REPO_URL = ''  # Route A, e.g. 'https://github.com/<user>/ViFeedback-NLP-Service.git'

WORK = pathlib.Path('/kaggle/working/vifeedback')
if WORK.exists():
    shutil.rmtree(WORK)

if REPO_URL:
    !git clone -q {REPO_URL} {WORK}
else:
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, (
        'No zip found under /kaggle/input. Add Data -> attach your vifeedback-repo dataset. '
        'See docs/KAGGLE_GUIDE.md step 1.'
    )
    print('using:', zips[0])
    WORK.mkdir(parents=True)
    !unzip -qo {zips[0]} -d {WORK}

os.chdir(WORK)
assert (WORK / 'pyproject.toml').exists(), f'pyproject.toml missing in {WORK} — bad archive?'
print('cwd:', pathlib.Path.cwd())
print(sorted(x.name for x in WORK.iterdir()))


### Install

`--no-deps` on the project itself: Kaggle already ships torch, numpy, pandas and sklearn, and
letting pip resolve `pyproject.toml` in full can pull a different torch and break CUDA.
The handful of packages Kaggle lacks are installed explicitly.


In [ ]:
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q typer pyyaml py-cpuinfo pyvi 2>&1 | tail -2
!pip install -q -U 'transformers>=4.44' 'datasets>=3.0' huggingface-hub 2>&1 | tail -2

import vifeedback

print('vifeedback', vifeedback.__version__)
!python -m vifeedback.cli --help | head -12


---

## 3. Data

Fetches UIT-VSFC and asserts the official split sizes (11,426 / 1,583 / 3,166) plus the leakage
figures. If the upstream ever shifts, this fails here rather than producing numbers against
different data.


In [ ]:
!python -m vifeedback.cli data fetch
!python -m pytest tests/data -q 2>&1 | tail -5


Build the segmentation variant. Segmentation is worth **+0.023 macro-F1** (ADR-012), so training
on raw text here would not be comparable to the laptop results. `pyvi` is the serving choice and
is pure Python, so no JVM is needed.


In [ ]:
!python -m vifeedback.cli data segmenters
!python -m vifeedback.cli data variants --name seg_pyvi


---

## 4. Train

**Runtime estimates** (P100, 5 seeds, 4 epochs). Kaggle sessions run up to 9 h and the free quota
is 30 GPU-hours/week, so run the cells you need rather than all of them.

| Cell | Model | Est. total |
|---|---|---|
| 4a | `phobert-large` | **~75 min** |
| 4b | `xlm-roberta-base` full | ~40 min |
| 4c | `CafeBERT` | ~90 min |

### On `--grad-accum`

`phobert-large` needs batch 16 to fit comfortably, but `phobert-base` was trained at 32. Comparing
them at different effective batch sizes would not be a controlled comparison, so `--grad-accum 2`
restores the effective batch to 32. The CLI prints the effective batch it is using — check it.


In [ ]:
# 4a — PhoBERT-large. The main reason this notebook exists (~7.9 GB).
!python -m vifeedback.cli train run \
    --task sentiment \
    --model phobert-large \
    --recipe base \
    --preprocessing seg_pyvi \
    --seeds all \
    --epochs 4 \
    --lr 1e-5 \
    --batch-size 16 \
    --grad-accum 2 \
    --max-length 96 \
    --phase 4


In [ ]:
# 4b — XLM-R base with UNFROZEN embeddings (~5.9 GB).
# The frozen-embedding version fits the laptop; run that one there, not here.
!python -m vifeedback.cli train run \
    --task sentiment \
    --model xlmr-base \
    --recipe base \
    --preprocessing seg_pyvi \
    --seeds all \
    --epochs 4 \
    --batch-size 32 \
    --max-length 96 \
    --phase 4


In [ ]:
# 4c — CafeBERT (XLM-R-large continued on 18GB Vietnamese, 560M params, ~11 GB).
# Needs registering in constants.MODEL_IDS first; skip unless you have added it.
# !python -m vifeedback.cli train run --task sentiment --model cafebert \
#     --preprocessing seg_pyvi --seeds all --batch-size 8 --grad-accum 4 --lr 8e-6 --phase 4


---

## 5. Verify before you leave

Confirms runs actually landed in the registry. A session that finished without writing rows has
produced nothing, and it is much cheaper to notice that here than after the session expires.


In [ ]:
import pandas as pd

from vifeedback.evaluation.report import load_registry

reg = load_registry()
new = reg[reg.run_id.str.contains('phobert-large|xlmr|cafebert', na=False)]
print(f'total registry rows: {len(reg)}   from this session: {len(new)}')
assert len(new) > 0, 'No new rows — the training cells did not produce anything.'
new[['run_id', 'task', 'model', 'split', 'macro_f1', 'weighted_f1', 'accuracy']]


In [ ]:
# Mean +/- std per configuration, the only form these numbers may be reported in.
if len(new):
    display(
        new[new.split == 'validation']
        .groupby(['task', 'model'])['macro_f1']
        .agg(['count', 'mean', 'std', 'min', 'max'])
        .round(4)
    )


---

## 6. Bring the results home

`/kaggle/working` is saved with the notebook version, so **Save Version → Output** already
persists everything. The zip below is for downloading it as one file.


In [ ]:
import shutil

shutil.make_archive('/kaggle/working/kaggle_results', 'zip', 'results')
sz = pathlib.Path('/kaggle/working/kaggle_results.zip').stat().st_size / 1e6
print(f'/kaggle/working/kaggle_results.zip  ({sz:.1f} MB)')
print('Download it from the Output panel on the right.')


### Merging on the laptop

Merge **by `run_id`**, never by blind append — the archive's `registry.csv` also contains rows
that were already committed before you built the dataset.

```bash
unzip -o kaggle_results.zip -d /tmp/kag
cp -rn /tmp/kag/runs/* results/runs/
python - <<'EOF'
import pandas as pd
a = pd.read_csv('results/registry.csv')
b = pd.read_csv('/tmp/kag/registry.csv')
out = pd.concat([a, b]).drop_duplicates(subset='run_id', keep='first')
out.to_csv('results/registry.csv', index=False)
print(f'{len(out) - len(a)} new rows merged')
EOF
```

### Reporting the split honestly

Rows produced here carry a different `env.json` — P100, not RTX 3050. That is irrelevant for
**accuracy**, which is hardware-independent, and disqualifying for **latency**, which is not.
Mark Kaggle-trained rows in the final results table and name the GPU.
